# 16) LLM-assisted CRN generation with a fake backend

This notebook demonstrates the modular LLM interface without calling VertexAI.  The default backend is a deterministic fake client, so the whole flow can be tested without credentials or billing.  Any future backend only needs to expose `generate_json(prompt, generation_config=None)`.

## 1) Imports and backend selection

Set `BACKEND = "fake"` for local testing.  Later, `BACKEND = "vertexai"` can be used once Google Cloud billing and credentials are available.

In [ ]:
import json
from pathlib import Path

import numpy as np

from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
    print_task_summary,
)
from RL4CRN.utils.crn_builders import build_logic_IOCRN
from RL4CRN.utils.library_builders import build_MAK_library
from RL4CRN.utils.default_tasks.LogicTaskKind import LogicTaskKind

from RL4CRN.llm import (
    LLMCandidateEvaluator,
    LLMCRNGenerator,
    LLMGenerationConfig,
    VertexLLMClient,
)

BACKEND = "fake"  # choices: "fake", "vertexai"

# Work whether the notebook is launched from the repository root or from apps/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "apps" else Path.cwd()
OUTPUT_DIR = REPO_ROOT / "comparisons" / "llm_crn_generation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2) Build a small RL4CRN task

The LLM layer does not define a separate CRN construction path.  It proposes reaction IDs and parameters, and the evaluator applies them through the same `Environment`, `LibraryActuator`, and `IOCRNStepper` used by RL.

In [ ]:
cfg = Configurator.preset("paper")
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-8
cfg.solver.atol = 1e-8

cfg.train.max_added_reactions = 3
cfg.train.batch_size = 4
cfg.train.n_cpus = 1
cfg.train.hall_of_fame_size = 10
cfg.train.seed = 0

n_inputs = 2
crn, species_labels = build_logic_IOCRN(
    n_inputs=n_inputs,
    include_dilution=False,
    solver=cfg.solver,
)
library_components = build_MAK_library(crn, species_labels, order=2)
library, M, K, masks = library_components

logic_fn = lambda x: all(x)  # AND gate
task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="logic",
    species_labels=species_labels,
    params={
        "n_inputs": n_inputs,
        "t_f": 40,
        "n_t": 300,
        "ic": ("constant", 0.01),
        "weights": "transient",
        "logic_fn": logic_fn,
    },
)

print_task_summary(task)
print("library size:", len(library))
print("reaction budget:", cfg.train.max_added_reactions)


## 3) Build the standard RL interfaces

We create a trainer only to reuse its `Session`, which already contains the template, library, task reward, actuator, stepper, and policy-ordering setting.

In [ ]:
trainer = make_session_and_trainer(cfg, task, logger=None)
session = trainer.s

evaluator = LLMCandidateEvaluator.from_session(session)
print("Evaluator ready. Ordered policy:", evaluator.is_ordered_policy)


## 4) Choose an LLM backend

The generator is backend-neutral.  A backend can be VertexAI, a local model, an HTTP service, or this fake client, as long as it returns a JSON-like object with a `candidates` list.

In [ ]:
class FakeLLMClient:
    """Deterministic local backend implementing the LLM client protocol."""

    def __init__(self, payload):
        self.payload = payload
        self.prompts = []

    def generate_json(self, prompt, generation_config=None):
        self.prompts.append(prompt)
        return self.payload


def make_fake_payload(library, max_added_reactions, num_candidates=3):
    """Create valid-shape candidates from the current reaction library."""

    candidates = []
    for offset in range(num_candidates):
        reaction_ids = [int((offset + j) % len(library)) for j in range(max_added_reactions)]
        parameter_values = []
        for rid in reaction_ids:
            n_params = int(library.get_reaction(rid).num_parameters)
            parameter_values.append([float(0.2 + 0.3 * (offset + 1))] * n_params)
        candidates.append(
            {
                "reasoning": f"fake candidate {offset}: deterministic backend smoke test",
                "reaction_ids": reaction_ids,
                "parameter_values": parameter_values,
            }
        )
    return {"candidates": candidates}


def make_llm_client(backend, *, library, max_added_reactions):
    """Backend factory. Future backends only need generate_json(...)."""

    if backend == "fake":
        return FakeLLMClient(
            make_fake_payload(
                library=library,
                max_added_reactions=max_added_reactions,
                num_candidates=4,
            )
        )

    if backend == "vertexai":
        return VertexLLMClient(
            project_id="YOUR_GOOGLE_CLOUD_PROJECT",
            location="europe-west1",
            model_name="gemini-2.5-flash",
        )

    raise ValueError(f"Unknown backend: {backend!r}")


client = make_llm_client(
    BACKEND,
    library=library,
    max_added_reactions=cfg.train.max_added_reactions,
)
print("Using backend:", BACKEND)


## 5) Generate and evaluate candidates

The fake client returns fixed candidates, but the rest of the path is identical to a real LLM backend: prompt construction, JSON parsing, candidate validation, CRN rollout, reward evaluation, feedback memory, Hall-of-Fame insertion, and JSONL logging.

In [ ]:
generator = LLMCRNGenerator(
    client=client,
    evaluator=evaluator,
    generation_config=LLMGenerationConfig(temperature=0.9),
)

task_description = "Design a CRN that implements a 2-input AND logic circuit with low transient loss."
jsonl_path = OUTPUT_DIR / "notebook16_fake_llm_candidates.jsonl"

round_result = generator.run_round(
    task_description=task_description,
    num_candidates=4,
    hall_of_fame_iter=session.mult_env.hall_of_fame,
    add_to_hall_of_fame=session.mult_env.hall_of_fame,
    jsonl_path=jsonl_path,
)

print("prompt characters:", len(round_result.prompt))
print("parsed candidates:", len(round_result.candidates))
print("evaluations logged to:", jsonl_path)

for i, evaluation in enumerate(round_result.evaluations):
    print(
        f"#{i}: valid={evaluation.valid} | loss={evaluation.loss} | message={evaluation.message}"
    )


## 6) Inspect accepted candidates

The accepted candidates were added to the same Hall of Fame used by RL training.  This makes it possible to compare, sample, or later use them for self-imitation style workflows.

In [ ]:
print("Hall of Fame size:", len(session.mult_env.hall_of_fame))

if len(session.mult_env.hall_of_fame) > 0:
    best_env = session.mult_env.hall_of_fame[0]
    print("Best LLM candidate loss:", best_env.state.last_task_info.get("reward"))
    print(best_env.state)

with jsonl_path.open(encoding="utf-8") as handle:
    first_record = json.loads(next(handle))

print("JSONL keys:", sorted(first_record.keys()))
print("First candidate IDs:", first_record["candidate"]["reaction_ids"])


## 7) Inspect the prompt sent to the backend

For real LLM runs, keep the prompt and JSONL outputs.  They are useful for reproducibility, debugging, and documenting AI-assisted design steps.

In [ ]:
print(round_result.prompt[:4000])
